In [1]:
import os
import glob
import torch
import numpy as np
import pandas as pd
import scanpy as sc


In [ ]:

# 4 个 pair 文件路径
pair_files = {
    "15vs0":  "/p1/data/jtian/SA/05_CAST/output_cast9/15_align_to_0_coords_final.data",
    "30vs15": "/p1/data/jtian/SA/05_CAST/output_cast9/30_align_to_15_coords_final.data",
    "45vs30": "/p1/data/jtian/SA/05_CAST/output_cast9/45_align_to_30_coords_final.data",
    "60vs45": "/p1/data/jtian/SA/05_CAST/output_cast9/60_align_to_45_coords_final.data",
}

# 你想要的 8 组列（每组会扁平化成 _x / _y 两列）
# (pair_name, sample_id_in_that_file, base_colname)
col_plan = [
    ("15vs0",  "0",  "0-15vs0"),
    ("15vs0",  "15", "15-15vs0"),

    ("30vs15", "15", "15-30vs15"),
    ("30vs15", "30", "30-30vs15"),

    ("45vs30", "30", "30-45vs30"),
    ("45vs30", "45", "45-45vs30"),

    ("60vs45", "45", "45-60vs45"),
    ("60vs45", "60", "60-60vs45"),
]

def to_numpy_xy(v):
    if torch.is_tensor(v):
        v = v.detach().cpu().numpy()
    else:
        v = np.asarray(v)
    return v  # 期望 (N,2)

# 1) 读入四个 dict（key 统一转 str）
loaded = {}
for pair_name, fp in pair_files.items():
    d = torch.load(fp, map_location="cpu")
    loaded[pair_name] = {str(k): v for k, v in d.items()}

# 2) 按顺序直接拼列（不做任何合并/覆盖；不做 KeyError 检查）
dfs = []
for pair_name, sample_id, base_col in col_plan:
    xy = to_numpy_xy(loaded[pair_name][str(sample_id)])
    df_xy = pd.DataFrame(xy, columns=[f"{base_col}_x", f"{base_col}_y"])
    dfs.append(df_xy)

merged = pd.concat(dfs, axis=1)

# 3) 保存
out_dir = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/"
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, "aligned_coords.csv")
merged.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print("Shape:", merged.shape)  # 预期列数 = 16
display(merged.head())


Saved: /p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords.csv
Shape: (53409, 16)


,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
0,49939.0,5370.0,50493.734375,-227.974762,43359.0,9530.0,43875.183594,9015.390625,29839.0,9230.0,29672.480469,9619.816406,15799.0,10650.0,15297.254883,10644.303711
1,49959.0,5370.0,50473.972656,-228.468903,43379.0,9530.0,43895.488281,9013.673828,29859.0,9230.0,29692.371094,9619.764648,15819.0,10650.0,15317.269531,10644.427734
2,49979.0,5370.0,50454.210938,-228.962799,43399.0,9530.0,43915.796875,9011.957031,29879.0,9230.0,29712.259766,9619.712891,15839.0,10650.0,15337.284180,10644.551758
3,49999.0,5370.0,50434.453125,-229.456940,43419.0,9530.0,43936.101562,9010.240234,29899.0,9230.0,29732.148438,9619.660156,15859.0,10650.0,15357.298828,10644.675781
4,50019.0,5370.0,50414.691406,-229.950836,43439.0,9530.0,43956.410156,9008.522461,29919.0,9230.0,29752.039062,9619.608398,15879.0,10650.0,15377.313477,10644.798828


In [2]:
display(merged.head(10))


,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
0,49939.0,5370.0,50493.734375,-227.974762,43359.0,9530.0,43875.183594,9015.390625,29839.0,9230.0,29672.480469,9619.816406,15799.0,10650.0,15297.254883,10644.303711
1,49959.0,5370.0,50473.972656,-228.468903,43379.0,9530.0,43895.488281,9013.673828,29859.0,9230.0,29692.371094,9619.764648,15819.0,10650.0,15317.269531,10644.427734
2,49979.0,5370.0,50454.210938,-228.962799,43399.0,9530.0,43915.796875,9011.957031,29879.0,9230.0,29712.259766,9619.712891,15839.0,10650.0,15337.284180,10644.551758
3,49999.0,5370.0,50434.453125,-229.456940,43419.0,9530.0,43936.101562,9010.240234,29899.0,9230.0,29732.148438,9619.660156,15859.0,10650.0,15357.298828,10644.675781
4,50019.0,5370.0,50414.691406,-229.950836,43439.0,9530.0,43956.410156,9008.522461,29919.0,9230.0,29752.039062,9619.608398,15879.0,10650.0,15377.313477,10644.798828
5,50039.0,5370.0,50394.929688,-230.444977,43459.0,9530.0,43976.714844,9006.805664,29939.0,9230.0,29771.927734,9619.556641,15899.0,10650.0,15397.328125,10644.922852
6,50059.0,5370.0,50375.167969,-230.939117,43479.0,9530.0,43997.023438,9005.088867,29959.0,9230.0,29791.816406,9619.504883,15919.0,10650.0,15417.342773,10645.046875
7,50079.0,5370.0,50355.406250,-231.433014,43499.0,9530.0,44017.328125,9003.372070,29979.0,9230.0,29811.707031,9619.452148,15939.0,10650.0,15437.357422,10645.170898
8,50099.0,5370.0,50335.644531,-231.927155,43519.0,9530.0,44037.632812,9001.655273,29999.0,9230.0,29831.595703,9619.400391,15959.0,10650.0,15457.372070,10645.294922
9,50119.0,5370.0,50315.886719,-232.421295,43539.0,9530.0,43649.933594,9012.034180,29619.0,9210.0,29513.316406,9601.032227,15639.0,10630.0,15477.386719,10645.418945


In [2]:
df = pd.read_csv("/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords.csv")
# 四舍五入
df = df.round(0)
display(df.head(10))
df.to_csv("/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int.csv", index=False)


,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
0,49939.0,5370.0,50494.0,-228.0,43359.0,9530.0,43875.0,9015.0,29839.0,9230.0,29672.0,9620.0,15799.0,10650.0,15297.0,10644.0
1,49959.0,5370.0,50474.0,-228.0,43379.0,9530.0,43895.0,9014.0,29859.0,9230.0,29692.0,9620.0,15819.0,10650.0,15317.0,10644.0
2,49979.0,5370.0,50454.0,-229.0,43399.0,9530.0,43916.0,9012.0,29879.0,9230.0,29712.0,9620.0,15839.0,10650.0,15337.0,10645.0
3,49999.0,5370.0,50434.0,-229.0,43419.0,9530.0,43936.0,9010.0,29899.0,9230.0,29732.0,9620.0,15859.0,10650.0,15357.0,10645.0
4,50019.0,5370.0,50415.0,-230.0,43439.0,9530.0,43956.0,9009.0,29919.0,9230.0,29752.0,9620.0,15879.0,10650.0,15377.0,10645.0
5,50039.0,5370.0,50395.0,-230.0,43459.0,9530.0,43977.0,9007.0,29939.0,9230.0,29772.0,9620.0,15899.0,10650.0,15397.0,10645.0
6,50059.0,5370.0,50375.0,-231.0,43479.0,9530.0,43997.0,9005.0,29959.0,9230.0,29792.0,9620.0,15919.0,10650.0,15417.0,10645.0
7,50079.0,5370.0,50355.0,-231.0,43499.0,9530.0,44017.0,9003.0,29979.0,9230.0,29812.0,9619.0,15939.0,10650.0,15437.0,10645.0
8,50099.0,5370.0,50336.0,-232.0,43519.0,9530.0,44038.0,9002.0,29999.0,9230.0,29832.0,9619.0,15959.0,10650.0,15457.0,10645.0
9,50119.0,5370.0,50316.0,-232.0,43539.0,9530.0,43650.0,9012.0,29619.0,9210.0,29513.0,9601.0,15639.0,10630.0,15477.0,10645.0


In [3]:
display(df.tail(10))

,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
53399,NaN,NaN,50002.0,5223.0,43719.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53400,NaN,NaN,49982.0,5222.0,43739.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53401,NaN,NaN,49962.0,5222.0,43759.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53402,NaN,NaN,49942.0,5221.0,43779.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53403,NaN,NaN,49922.0,5221.0,43799.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53404,NaN,NaN,49903.0,5220.0,43819.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,49883.0,5220.0,43839.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,49863.0,5219.0,43859.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,49843.0,5219.0,43879.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53408,NaN,NaN,50041.0,5242.0,43679.0,3530.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

import re
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ========== 1) 解析列名 ==========
PAT = re.compile(r'^(?P<sample>[^-]+)-(?P<group>[^_]+)_(?P<axis>[xy])$')

def sort_key(sample: str):
    # 样本名如果是数字（如 0,15,30...），按数值排序；否则按字符串排序
    s = str(sample)
    if re.fullmatch(r'-?\d+(\.\d+)?', s):
        return float(s)
    return s

# ========== 2) 组内最近邻匹配（贪心一对一）==========
def align_group(df: pd.DataFrame, left_cols, right_cols, k=10):
    """
    left_cols = (left_x_col, left_y_col)
    right_cols = (right_x_col, right_y_col)
    返回：对齐后的 right_x, right_y, orig_idx（长度=全表行数）
    """
    lx, ly = left_cols
    rx, ry = right_cols

    left_xy = df[[lx, ly]].to_numpy()
    right_xy = df[[rx, ry]].to_numpy()

    left_valid = np.isfinite(left_xy).all(axis=1)
    right_valid = np.isfinite(right_xy).all(axis=1)

    left_rows = np.where(left_valid)[0]
    right_rows = np.where(right_valid)[0]

    L = left_xy[left_valid]
    R = right_xy[right_valid]

    aligned_rx = np.full(len(df), np.nan)
    aligned_ry = np.full(len(df), np.nan)
    orig_idx   = np.full(len(df), -1, dtype=int)

    if len(L) == 0 or len(R) == 0:
        return aligned_rx, aligned_ry, orig_idx

    # KDTree: 右点 -> 左点 的 k 个最近邻候选
    tree = cKDTree(L)
    dists, neigh = tree.query(R, k=min(k, len(L)))

    if dists.ndim == 1:
        dists = dists[:, None]
        neigh = neigh[:, None]

    # 构造候选边 (dist, r_idx, l_idx)，按距离排序，贪心保证“一对一”
    candidates = []
    for rr in range(len(R)):
        for j in range(neigh.shape[1]):
            ll = int(neigh[rr, j])
            dist = float(dists[rr, j])
            if np.isfinite(dist):
                candidates.append((dist, rr, ll))
    candidates.sort(key=lambda x: x[0])

    left_used  = np.zeros(len(L), dtype=bool)
    right_used = np.zeros(len(R), dtype=bool)
    left_to_right = np.full(len(L), -1, dtype=int)

    for dist, rr, ll in candidates:
        if (not left_used[ll]) and (not right_used[rr]):
            left_used[ll] = True
            right_used[rr] = True
            left_to_right[ll] = rr

    # 写回：右样本点搬到左样本对应行，并记录右样本原行索引
    for ll, rr in enumerate(left_to_right):
        if rr == -1:
            continue
        full_left_row  = left_rows[ll]
        full_right_row = right_rows[rr]

        aligned_rx[full_left_row] = right_xy[full_right_row, 0]
        aligned_ry[full_left_row] = right_xy[full_right_row, 1]
        orig_idx[full_left_row]   = int(df.index[full_right_row])  # 原行索引（0..n-1）

    return aligned_rx, aligned_ry, orig_idx

# ========== 3) 主流程 ==========
def align_csv(input_csv: str, output_csv: str, k=10):
    df = pd.read_csv(input_csv)

    # 收集 group -> sample -> {xcol,ycol}
    groups = {}
    for col in df.columns:
        m = PAT.match(col)
        if not m:
            continue
        sample, group, axis = m.group("sample"), m.group("group"), m.group("axis")
        groups.setdefault(group, {}).setdefault(sample, {})[axis] = col

    df_out = df.copy()

    # 对每个组别做“右对齐到左”
    for group, samples_dict in groups.items():
        samples = sorted(samples_dict.keys(), key=sort_key)
        if len(samples) != 2:
            # 如果某组不是恰好两个样本，这里跳过（也可以自己扩展）
            continue

        left, right = samples[0], samples[1]
        lx, ly = samples_dict[left]["x"], samples_dict[left]["y"]
        rx, ry = samples_dict[right]["x"], samples_dict[right]["y"]

        aligned_rx, aligned_ry, orig_idx = align_group(df_out, (lx, ly), (rx, ry), k=k)

        df_out[rx] = aligned_rx
        df_out[ry] = aligned_ry

        orig_col = f"{right}-{group}_orig_idx"
        df_out[orig_col] = orig_idx

    # 把 orig_idx 列插到对应右样本 y 后面（更好看）
    new_cols = []
    for col in df.columns:
        new_cols.append(col)
        m = PAT.match(col)
        if m and m.group("axis") == "y":
            sample, group = m.group("sample"), m.group("group")
            sd = groups.get(group, {})
            if len(sd) == 2:
                samples = sorted(sd.keys(), key=sort_key)
                right = samples[1]
                if sample == right:
                    orig_col = f"{right}-{group}_orig_idx"
                    if orig_col in df_out.columns:
                        new_cols.append(orig_col)

    # 追加任何漏掉的新列
    for c in df_out.columns:
        if c not in new_cols:
            new_cols.append(c)

    df_out = df_out[new_cols]
    df_out.to_csv(output_csv, index=False)

# ========== 4) 运行示例 ==========
if __name__ == "__main__":
    align_csv("/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int.csv", "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int_match.csv", k=50)



In [6]:
import re
import numpy as np
import pandas as pd

in_csv  = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int_match.csv"
out_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv"

df = pd.read_csv(in_csv)

PAT = re.compile(r'^(?P<sample>[^-]+)-(?P<group>[^_]+)_(?P<axis>[xy])$')

def sort_key(sample: str):
    s = str(sample)
    if re.fullmatch(r'-?\d+(\.\d+)?', s):
        return float(s)
    return s

# --------- parse groups / columns ----------
groups = {}
for col in df.columns:
    m = PAT.match(col)
    if not m:
        continue
    sample, group, axis = m.group("sample"), m.group("group"), m.group("axis")
    groups.setdefault(group, {}).setdefault(sample, {})[axis] = col

for col in df.columns:
    if col.endswith("_orig_idx"):
        pre = col[:-len("_orig_idx")]
        if "-" in pre:
            sample, group = pre.split("-", 1)
            groups.setdefault(group, {}).setdefault(sample, {})["orig_idx"] = col

# --------- build ordered adjacent pairs (0-15, 15-30, 30-45, 45-60) ----------
pairs = []
for group, sd in groups.items():
    samples = sorted(sd.keys(), key=sort_key)
    if len(samples) != 2:
        continue
    left, right = samples[0], samples[1]
    pairs.append((float(left), float(right), group, sd))
pairs = sorted(pairs, key=lambda t: t[1])  # by right sample

pair_infos = []
for left, right, group, sd in pairs:
    left = str(int(left)); right = str(int(right))
    info = dict(
        left=left, right=right, group=group,
        lx=sd[left]["x"], ly=sd[left]["y"],
        rx=sd[right]["x"], ry=sd[right]["y"],
        orig=sd[right]["orig_idx"]
    )
    # map: (right original row idx) -> (left row idx in df)
    valid = df[[info["rx"], info["ry"]]].notna().all(axis=1) & (df[info["orig"]].fillna(-1).astype(int) != -1)
    right_idx = df.loc[valid, info["orig"]].astype(int).to_numpy()
    left_row  = df.index[valid].to_numpy().astype(int)

    m = {}
    for ri, lr in zip(right_idx, left_row):
        if ri not in m:
            m[ri] = lr
    info["map"] = pd.Series(m)  # index=right_idx, value=left_row
    info["map"].index = info["map"].index.astype(int)

    pair_infos.append(info)

# expect 4 pairs
pi01, pi1530, pi3045, pi4560 = pair_infos

# --------- chain from sample60 backward ----------
valid_last = df[[pi4560["rx"], pi4560["ry"]]].notna().all(axis=1) & (df[pi4560["orig"]].fillna(-1).astype(int) != -1)

idx45 = df.index[valid_last].to_numpy().astype(int)                  # sample45 row idx
idx60 = df.loc[valid_last, pi4560["orig"]].astype(int).to_numpy()    # sample60 orig idx

idx30 = pi3045["map"].reindex(idx45).to_numpy()
idx15 = pi1530["map"].reindex(idx30).to_numpy()
idx0  = pi01["map"].reindex(idx15).to_numpy()

mask = np.isfinite(idx30) & np.isfinite(idx15) & np.isfinite(idx0)

idx0  = idx0[mask].astype(int)
idx15 = idx15[mask].astype(int)
idx30 = idx30[mask].astype(int)
idx45 = idx45[mask].astype(int)
idx60 = idx60[mask].astype(int)

# --------- output (A0,A15,A30,A45,A60 on one row) ----------
out = pd.DataFrame({
    "idx_0":  idx0,
    "x_0":    df.loc[idx0,  pi01["lx"]].to_numpy(),
    "y_0":    df.loc[idx0,  pi01["ly"]].to_numpy(),

    "idx_15": idx15,
    "x_15":   df.loc[idx15, pi1530["lx"]].to_numpy(),
    "y_15":   df.loc[idx15, pi1530["ly"]].to_numpy(),

    "idx_30": idx30,
    "x_30":   df.loc[idx30, pi3045["lx"]].to_numpy(),
    "y_30":   df.loc[idx30, pi3045["ly"]].to_numpy(),

    "idx_45": idx45,
    "x_45":   df.loc[idx45, pi4560["lx"]].to_numpy(),
    "y_45":   df.loc[idx45, pi4560["ly"]].to_numpy(),

    "idx_60": idx60,
    "x_60":   df.loc[idx45, pi4560["rx"]].to_numpy(),  # 60点被移到45点所在行
    "y_60":   df.loc[idx45, pi4560["ry"]].to_numpy(),
})

out.insert(0, "chain_id", np.arange(len(out), dtype=int))
out.to_csv(out_csv, index=False)

print("saved:", out_csv, "chains:", len(out))


saved: /p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv chains: 37936


In [2]:
# 输入 / 输出文件
in_csv  = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv"
out_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv"

# 读取
df = pd.read_csv(in_csv)

print("===== BEFORE DROP (head) =====")
print(df.head())
print("\nColumns:", list(df.columns))

# 删除 chain_id 和所有 idx_* 列
cols_to_drop = ["chain_id"] + [c for c in df.columns if c.startswith("idx_")]
df_out = df.drop(columns=cols_to_drop, errors="ignore")

print("\n===== AFTER DROP (head) =====")
print(df_out.head())
print("\nColumns:", list(df_out.columns))




===== BEFORE DROP (head) =====
   chain_id  idx_0      x_0    y_0  idx_15     x_15    y_15  idx_30     x_30  \
0         0  46607  49959.0  230.0    2494  43879.0  9010.0       0  29839.0   
1         1  46606  49939.0  230.0    2495  43899.0  9010.0       1  29859.0   
2         2  46605  49919.0  230.0    2497  43939.0  9010.0       3  29899.0   
3         3  46604  49899.0  230.0    2498  43959.0  9010.0       4  29919.0   
4         4  46603  49879.0  230.0    2499  43979.0  9010.0       5  29939.0   

     y_30  idx_45     x_45     y_45  idx_60     x_60     y_60  
0  9230.0    1646  15959.0  10250.0    1800  15960.0  10244.0  
1  9230.0    1647  15979.0  10250.0    1801  15980.0  10244.0  
2  9230.0    1649  16019.0  10250.0    1803  16020.0  10244.0  
3  9230.0    1650  16039.0  10250.0    1804  16040.0  10244.0  
4  9230.0    1651  16059.0  10250.0    1805  16060.0  10245.0  

Columns: ['chain_id', 'idx_0', 'x_0', 'y_0', 'idx_15', 'x_15', 'y_15', 'idx_30', 'x_30', 'y_30', 'idx_4

In [3]:
# 保存
df_out.to_csv(out_csv, index=False)
print("\nSaved to:", out_csv)


Saved to: /p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv


In [6]:
adata = sc.read_h5ad("/p1/data/jtian/SA/05_CAST/output_cast9/demo1_cast9.h5ad")
originCoords60 = adata.obs.loc[adata.obs['sample']=='60'].copy()
print(originCoords60.head())
print(originCoords60.columns)


              x      y sample
SpotIndex                    
196698     1899  10530     60
196699     1919  10530     60
196700     1939  10530     60
196701     1959  10530     60
196702     1979  10530     60
Index(['x', 'y', 'sample'], dtype='object')


In [7]:
originCoords60_xy =originCoords60[['x','y']].reset_index(drop=True).rename(columns={'x':"60-origin_x", "y": "60-origin_y"})
print(originCoords60_xy.head())

   60-origin_x  60-origin_y
0         1899        10530
1         1919        10530
2         1939        10530
3         1959        10530
4         1979        10530


In [8]:
# 读入主表
main_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int.csv"
df_main = pd.read_csv(main_csv)

print("===== MAIN CSV (head) =====")
print(df_main.head())
print("main shape:", df_main.shape)

print("\n===== originCoords60_xy (head) =====")
print(originCoords60_xy.head())
print("originCoords60_xy shape:", originCoords60_xy.shape)

# 直接按行拼到右侧，自动补 NA
df_merged = pd.concat(
    [df_main.reset_index(drop=True),
     originCoords60_xy.reset_index(drop=True)],
    axis=1
)

print("\n===== MERGED (head) =====")
print(df_merged.head())
print("merged shape:", df_merged.shape)

# 保存
out_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int_with_origin60.csv"
df_merged.to_csv(out_csv, index=False)

print("\nSaved to:", out_csv)


===== MAIN CSV (head) =====
   0-15vs0_x  0-15vs0_y  15-15vs0_x  15-15vs0_y  15-30vs15_x  15-30vs15_y  \
0    49939.0     5370.0     50494.0      -228.0      43359.0       9530.0   
1    49959.0     5370.0     50474.0      -228.0      43379.0       9530.0   
2    49979.0     5370.0     50454.0      -229.0      43399.0       9530.0   
3    49999.0     5370.0     50434.0      -229.0      43419.0       9530.0   
4    50019.0     5370.0     50415.0      -230.0      43439.0       9530.0   

   30-30vs15_x  30-30vs15_y  30-45vs30_x  30-45vs30_y  45-45vs30_x  \
0      43875.0       9015.0      29839.0       9230.0      29672.0   
1      43895.0       9014.0      29859.0       9230.0      29692.0   
2      43916.0       9012.0      29879.0       9230.0      29712.0   
3      43936.0       9010.0      29899.0       9230.0      29732.0   
4      43956.0       9009.0      29919.0       9230.0      29752.0   

   45-45vs30_y  45-60vs45_x  45-60vs45_y  60-60vs45_x  60-60vs45_y  
0       9620.0     

In [9]:
# ---------------- paths ----------------
chained_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60.csv"
aligned_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int_with_origin60.csv"
out_csv     = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_with_origin60.csv"

# ---------------- load ----------------
df_chain = pd.read_csv(chained_csv)
df_align = pd.read_csv(aligned_csv)

print("===== chained BEFORE (head) =====")
print(df_chain.head())

print("\n===== aligned (head) =====")
print(df_align.head())

# ---------------- sanity checks ----------------
need_chain = ["x_60", "y_60"]
need_align = ["60-60vs45_x", "60-60vs45_y", "60-origin_x", "60-origin_y"]
for c in need_chain:
    if c not in df_chain.columns:
        raise KeyError(f"Missing in chained: {c}")
for c in need_align:
    if c not in df_align.columns:
        raise KeyError(f"Missing in aligned: {c}")

# ---------------- prepare aligned lookup table ----------------
# 保存 aligned 的“行索引”（行号），便于回查
df_align2 = df_align.reset_index(drop=False).rename(columns={"index": "aligned_row_idx"})

# 只保留用于匹配与回填的列
df_lookup = df_align2[[
    "aligned_row_idx",
    "60-60vs45_x", "60-60vs45_y",
    "60-origin_x", "60-origin_y"
]].copy()

# 去掉匹配坐标缺失的行（否则无法匹配）
df_lookup = df_lookup.dropna(subset=["60-60vs45_x", "60-60vs45_y"])

# 如果同一坐标在 aligned 里出现多次：保留第一条（你也可以改成保留最后一条）
df_lookup = df_lookup.drop_duplicates(subset=["60-60vs45_x", "60-60vs45_y"], keep="first")

# ---------------- merge: chain x_60/y_60  <-> aligned 60-60vs45_x/y ----------------
df_chain2 = df_chain.copy()
df_chain2["__chain_row__"] = range(len(df_chain2))  # 保序

df_merged = df_chain2.merge(
    df_lookup,
    how="left",
    left_on=["x_60", "y_60"],
    right_on=["60-60vs45_x", "60-60vs45_y"],
    suffixes=("", "_from_aligned")
)

# ---------------- stats ----------------
matched = df_merged["aligned_row_idx"].notna().sum()
total = len(df_merged)
print(f"\nMatch stats: {matched}/{total} matched ({matched/total:.2%})")

# ---------------- write back columns ----------------
# 把 aligned 的 origin 坐标加到 chained 对应行
df_out = df_chain.copy()
df_out["60-origin_x"] = df_merged["60-origin_x"].to_numpy()
df_out["60-origin_y"] = df_merged["60-origin_y"].to_numpy()
# 可选：也把 aligned 行号带回去方便检查
df_out["aligned_row_idx_for_60"] = df_merged["aligned_row_idx"].to_numpy()

print("\n===== chained AFTER (head) =====")
print(df_out.head())

# ---------------- save ----------------
df_out.to_csv(out_csv, index=False)
print("\nSaved to:", out_csv)


===== chained BEFORE (head) =====
       x_0    y_0     x_15    y_15     x_30    y_30     x_45     y_45  \
0  49959.0  230.0  43879.0  9010.0  29839.0  9230.0  15959.0  10250.0   
1  49939.0  230.0  43899.0  9010.0  29859.0  9230.0  15979.0  10250.0   
2  49919.0  230.0  43939.0  9010.0  29899.0  9230.0  16019.0  10250.0   
3  49899.0  230.0  43959.0  9010.0  29919.0  9230.0  16039.0  10250.0   
4  49879.0  230.0  43979.0  9010.0  29939.0  9230.0  16059.0  10250.0   

      x_60     y_60  
0  15960.0  10244.0  
1  15980.0  10244.0  
2  16020.0  10244.0  
3  16040.0  10244.0  
4  16060.0  10245.0  

===== aligned (head) =====
   0-15vs0_x  0-15vs0_y  15-15vs0_x  15-15vs0_y  15-30vs15_x  15-30vs15_y  \
0    49939.0     5370.0     50494.0      -228.0      43359.0       9530.0   
1    49959.0     5370.0     50474.0      -228.0      43379.0       9530.0   
2    49979.0     5370.0     50454.0      -229.0      43399.0       9530.0   
3    49999.0     5370.0     50434.0      -229.0      43419.

In [10]:
# ---------- paths ----------
in_csv  = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_with_origin60.csv"
out_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_replace60.csv"

# ---------- load ----------
df = pd.read_csv(in_csv)

print("===== BEFORE REPLACE (head) =====")
print(df[["x_60", "y_60", "60-origin_x", "60-origin_y"]].head())

# ---------- replace (row-wise) ----------
mask = df["60-origin_x"].notna() & df["60-origin_y"].notna()

df.loc[mask, "x_60"] = df.loc[mask, "60-origin_x"]
df.loc[mask, "y_60"] = df.loc[mask, "60-origin_y"]

print("\n===== AFTER REPLACE (head) =====")
print(df[["x_60", "y_60", "60-origin_x", "60-origin_y"]].head())

print(f"\nReplaced rows: {mask.sum()} / {len(df)}")

# ---------- save ----------
df.to_csv(out_csv, index=False)
print("\nSaved to:", out_csv)


===== BEFORE REPLACE (head) =====
      x_60     y_60  60-origin_x  60-origin_y
0  15960.0  10244.0       2559.0      10130.0
1  15980.0  10244.0       2579.0      10130.0
2  16020.0  10244.0       2619.0      10130.0
3  16040.0  10244.0       2639.0      10130.0
4  16060.0  10245.0       2659.0      10130.0

===== AFTER REPLACE (head) =====
     x_60     y_60  60-origin_x  60-origin_y
0  2559.0  10130.0       2559.0      10130.0
1  2579.0  10130.0       2579.0      10130.0
2  2619.0  10130.0       2619.0      10130.0
3  2639.0  10130.0       2639.0      10130.0
4  2659.0  10130.0       2659.0      10130.0

Replaced rows: 37936 / 37936

Saved to: /p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_replace60.csv


In [12]:
# ---------- paths ----------
in_csv  = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_replace60.csv"
out_csv = "/p1/data/jtian/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_final.csv"

# ---------- load ----------
df = pd.read_csv(in_csv)

print("===== BEFORE DROP (head) =====")
print(df.head())
print("\nColumns BEFORE:", list(df.columns))

# ---------- drop columns ----------
df_out = df.drop(columns=["60-origin_x", "60-origin_y","aligned_row_idx_for_60"], errors="ignore")

print("\n===== AFTER DROP (head) =====")
print(df_out.head())
print("\nColumns AFTER:", list(df_out.columns))

# ---------- save ----------
df_out.to_csv(out_csv, index=False)
print("\nSaved to:", out_csv)


===== BEFORE DROP (head) =====
       x_0    y_0     x_15    y_15     x_30    y_30     x_45     y_45    x_60  \
0  49959.0  230.0  43879.0  9010.0  29839.0  9230.0  15959.0  10250.0  2559.0   
1  49939.0  230.0  43899.0  9010.0  29859.0  9230.0  15979.0  10250.0  2579.0   
2  49919.0  230.0  43939.0  9010.0  29899.0  9230.0  16019.0  10250.0  2619.0   
3  49899.0  230.0  43959.0  9010.0  29919.0  9230.0  16039.0  10250.0  2639.0   
4  49879.0  230.0  43979.0  9010.0  29939.0  9230.0  16059.0  10250.0  2659.0   

      y_60  60-origin_x  60-origin_y  aligned_row_idx_for_60  
0  10130.0       2559.0      10130.0                    1800  
1  10130.0       2579.0      10130.0                    1801  
2  10130.0       2619.0      10130.0                    1803  
3  10130.0       2639.0      10130.0                    1804  
4  10130.0       2659.0      10130.0                    1805  

Columns BEFORE: ['x_0', 'y_0', 'x_15', 'y_15', 'x_30', 'y_30', 'x_45', 'y_45', 'x_60', 'y_60', '60-orig

In [2]:
coordsMerge=pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_final.csv")

In [3]:
coordsMerge

,x_0,y_0,x_15,y_15,x_30,y_30,x_45,y_45,x_60,y_60
0,49959.0,230.0,43879.0,9010.0,29839.0,9230.0,15959.0,10250.0,2559.0,10130.0
1,49939.0,230.0,43899.0,9010.0,29859.0,9230.0,15979.0,10250.0,2579.0,10130.0
2,49919.0,230.0,43939.0,9010.0,29899.0,9230.0,16019.0,10250.0,2619.0,10130.0
3,49899.0,230.0,43959.0,9010.0,29919.0,9230.0,16039.0,10250.0,2639.0,10130.0
4,49879.0,230.0,43979.0,9010.0,29939.0,9230.0,16059.0,10250.0,2659.0,10130.0
...,...,...,...,...,...,...,...,...,...,...
37931,49839.0,5130.0,43879.0,3650.0,30299.0,4330.0,16439.0,5150.0,2999.0,5090.0
37932,50139.0,5150.0,43579.0,3630.0,29999.0,4310.0,16139.0,5130.0,2699.0,5070.0
37933,50119.0,5130.0,43599.0,3650.0,30019.0,4310.0,16159.0,5130.0,2719.0,5070.0
37934,50099.0,5130.0,43619.0,3650.0,30039.0,4310.0,16179.0,5130.0,2739.0,5070.0


In [7]:
print(type(coordsMerge.iloc[0,0]))

<class 'numpy.float64'>


In [12]:
coordsMerge=coordsMerge.astype(int)

In [13]:
print(type(coordsMerge.iloc[0,0]))

<class 'numpy.int64'>


In [14]:
coordsMerge.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/chained_0_15_30_45_60_final.csv",index=False)

In [4]:
adata=sc.read_h5ad("/p2/zulab/jtian/data/SA/05_CAST/output_cast9/demo1_cast9.h5ad")

In [5]:
adata.obs

,x,y,sample
SpotIndex,,,
1,49939,5370,0
2,49959,5370,0
3,49979,5370,0
4,49999,5370,0
5,50019,5370,0
...,...,...,...
247357,3119,4970,60
247358,3139,4970,60
247359,3159,4970,60


In [10]:
type(adata.obs['x'][0])

/tmp/ipykernel_1571149/3335257290.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  type(adata.obs['x'][0])


numpy.int64

In [16]:
SAMPLE_COL = 'sample'
X_COL = 'x'
Y_COL = 'y'
samples = [0, 15, 30, 45, 60]
obs = adata.obs.copy()
obs['SpotIndex'] = obs.index
new_cols = []
for s in samples:
    xcol, ycol = f'x_{s}', f'y_{s}'
    idxcol = f'SpotIndex_{s}'
    sub = obs.loc[obs[SAMPLE_COL].astype(str) == str(s), [X_COL, Y_COL, 'SpotIndex']]
    mapper = dict(zip(zip(sub[X_COL].to_numpy(), sub[Y_COL].to_numpy()), sub['SpotIndex'].to_numpy()))
    coordsMerge[idxcol] = [mapper.get((x, y), pd.NA) for x, y in zip(coordsMerge[xcol], coordsMerge[ycol])]
    new_cols += [idxcol]
    miss = coordsMerge[idxcol].isna().sum()
    print(f"sample {s}: matched {len(coordsMerge)-miss}/{len(coordsMerge)} missing={miss}")
print(new_cols)


sample 0: matched 37936/37936 missing=0
sample 15: matched 37936/37936 missing=0
sample 30: matched 37936/37936 missing=0
sample 45: matched 37936/37936 missing=0
sample 60: matched 37936/37936 missing=0
['SpotIndex_0', 'SpotIndex_15', 'SpotIndex_30', 'SpotIndex_45', 'SpotIndex_60']


In [22]:
indexMerge=coordsMerge[new_cols].copy()
print(indexMerge)


      SpotIndex_0 SpotIndex_15 SpotIndex_30 SpotIndex_45 SpotIndex_60
0           46608        49890       100805       147668       198498
1           46607        49891       100806       147669       198499
2           46606        49893       100808       147671       198501
3           46605        49894       100809       147672       198502
4           46604        49895       100810       147673       198503
...           ...          ...          ...          ...          ...
37931         849       100545       145640       196444       247057
37932         770       100602       145690       196484       247110
37933         863       100531       145691       196485       247111
37934         862       100532       145692       196486       247112
37935         343       100804       145695       196489       247115

[37936 rows x 5 columns]


In [23]:
indexMerge.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/alignedIndexMerge.csv",index=False)
coordsMerge.to_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/alignedCoordsIndexMerge.csv",index=False)

In [24]:
indexMerge=pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/alignedIndexMerge.csv")

In [25]:
indexMerge

,SpotIndex_0,SpotIndex_15,SpotIndex_30,SpotIndex_45,SpotIndex_60
0,46608,49890,100805,147668,198498
1,46607,49891,100806,147669,198499
2,46606,49893,100808,147671,198501
3,46605,49894,100809,147672,198502
4,46604,49895,100810,147673,198503
...,...,...,...,...,...
37931,849,100545,145640,196444,247057
37932,770,100602,145690,196484,247110
37933,863,100531,145691,196485,247111
37934,862,100532,145692,196486,247112


In [33]:
intensity0 = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/input/MetaboliteIntensity/01-intensityRMS.csv", sep=";", header = 0, index_col = 0)
intensity0 = intensity0.T
intensity0.index = intensity0.index.str.replace('Spot','',regex=False).str.strip().astype(int)

intensity15 = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/input/MetaboliteIntensity/02-intensityRMS.csv", sep=";", header = 0, index_col = 0)
intensity15 = intensity15.T
intensity15.index = intensity15.index.str.replace('Spot','',regex=False).str.strip().astype(int)

intensity30 = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/input/MetaboliteIntensity/03-intensityRMS.csv", sep=";", header=0, index_col=0)
intensity30 = intensity30.T
intensity30.index = intensity30.index.str.replace('Spot', '', regex=False).str.strip().astype(int)

intensity45 = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/input/MetaboliteIntensity/04-intensityRMS.csv", sep=";", header=0, index_col=0)
intensity45 = intensity45.T
intensity45.index = intensity45.index.str.replace('Spot', '', regex=False).str.strip().astype(int)

intensity60 = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/input/MetaboliteIntensity/05-intensityRMS.csv", sep=";", header=0, index_col=0)
intensity60 = intensity60.T
intensity60.index = intensity60.index.str.replace('Spot', '', regex=False).str.strip().astype(int)


In [34]:
intensity0

m/z,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,72.0170,73.0294,74.0246,...,856.5052,859.5297,860.5360,863.5618,864.5684,882.5853,921.5991,923.6150,985.6046,986.6105
1,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,0.354188,0.563068,0.000000,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
2,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,0.000000,0.875714,0.197465,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
3,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,0.000000,0.000000,0.510047,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
4,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,0.143601,0.574405,0.312544,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,0.000000,1.460180,0.499793,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47391,0.000000,0.717167,4.328612,0.640327,16.008182,0.947684,2.177113,0.947684,4.123708,0.000000,...,3.970029,5.071392,3.201636,2.868666,3.022345,0.000000,3.739511,2.638148,1.536786,0.000000
47392,0.998003,0.000000,3.669128,3.404951,22.132178,0.000000,0.000000,0.000000,5.312897,0.000000,...,1.819887,4.344247,4.080070,3.639775,1.350239,2.729831,0.000000,1.262180,2.612419,1.731828
47393,0.954880,1.432320,4.190861,0.981404,12.360387,0.000000,2.466773,1.777137,5.517083,3.182932,...,2.201528,4.005190,2.811590,6.657633,0.000000,0.000000,1.379271,1.697564,1.644515,1.273173
47394,1.149264,2.667935,7.798578,1.477625,23.518871,0.000000,1.231354,0.000000,3.611973,0.000000,...,3.406747,2.832115,4.022425,10.138152,4.186605,3.406747,1.723896,3.406747,0.369406,0.656722


In [39]:
intensity0.columns.name = None
intensity15.columns.name = None
intensity30.columns.name = None
intensity45.columns.name = None
intensity60.columns.name = None

#打印形状 (Shape)
print("--- 数据形状 (Shape) ---")
print(intensity0.shape)
print(intensity15.shape)
print(intensity30.shape)
print(intensity45.shape)
print(intensity60.shape)

#打印前几行数据 (Head)
print("\n--- 数据预览 (Head) ---")
print("样本 0:")
print(intensity0.head())
print("\n样本 15:")
print(intensity15.head())
print("\n样本 30:")
print(intensity30.head())
print("\n样本 45:")
print(intensity45.head())
print("\n样本 60:")
print(intensity60.head())

--- 数据形状 (Shape) ---
(47395, 696)
(53409, 696)
(45217, 696)
(50676, 696)
(50664, 696)

--- 数据预览 (Head) ---
样本 0:
   57.0346   58.0296   59.0139   67.0191   71.0137   71.0503   72.0091   \
1  0.208880  0.254289  0.499496  0.399597  1.607469  0.127144  0.962665   
2  0.000000  0.223221  0.480784  0.257563  2.481191  0.000000  1.210547   
3  0.000000  0.000000  1.175326  0.720719  0.698543  0.000000  0.155232   
4  0.152048  0.000000  0.557510  0.000000  0.726453  0.000000  0.278755   
5  0.000000  0.000000  0.725190  0.382195  2.587164  0.000000  0.000000   

   72.0170   73.0294   74.0246   ...  856.5052  859.5297  860.5360  863.5618  \
1  0.354188  0.563068  0.000000  ...  0.363270  0.481332  0.163471  0.000000   
2  0.000000  0.875714  0.197465  ...  0.712591  0.154538  0.283319  0.394930   
3  0.000000  0.000000  0.510047  ...  0.177408  0.155232  0.221760  0.410255   
4  0.143601  0.574405  0.312544  ...  0.084471  0.000000  0.194284  0.304097   
5  0.000000  1.460180  0.499793  ...

In [ ]:
SAMPLE_COL = 'sample'
X_COL = 'x'
Y_COL = 'y'
samples = [0, 15, 30, 45, 60]
obs = adata.obs.copy()
obs['SpotIndex'] = obs.index
new_cols = []
for s in samples:
    xcol, ycol = f'x_{s}', f'y_{s}'
    idxcol = f'SpotIndex_{s}'
    sub = obs.loc[obs[SAMPLE_COL].astype(str) == str(s), [X_COL, Y_COL, 'SpotIndex']]
    mapper = dict(zip(zip(sub[X_COL].to_numpy(), sub[Y_COL].to_numpy()), sub['SpotIndex'].to_numpy()))
    coordsMerge[idxcol] = [mapper.get((x, y), pd.NA) for x, y in zip(coordsMerge[xcol], coordsMerge[ycol])]
    new_cols += [idxcol]
    miss = coordsMerge[idxcol].isna().sum()
    print(f"sample {s}: matched {len(coordsMerge)-miss}/{len(coordsMerge)} missing={miss}")
print(new_cols)

In [42]:
IntenCols = intensity0.columns

In [44]:
Mindex = indexMerge.index

In [ ]:
import numpy as np

# x: intensity（填你的5个值）


# y: 样本值（固定就是0,15,30,45,60）


# 拟合 y = a*x + b


# 预测与R^2
   # 截距就是x=0的y


In [49]:
cols = [f'{Icol}-RE' for Icol in IntenCols] + \
       [f'{Icol}-r2' for Icol in IntenCols] + \
       [f'{Icol}-yValue' for Icol in IntenCols]
Mindex = range(len(indexMerge))
result_df = pd.DataFrame(index=Mindex, columns=cols, dtype=float)
samples = np.array([0,15,30,45,60], dtype=float)

for Icol in IntenCols:
    for mindex in Mindex:
        s0  = indexMerge['SpotIndex_0'].iloc[mindex]
        s15 = indexMerge['SpotIndex_15'].iloc[mindex]
        s30 = indexMerge['SpotIndex_30'].iloc[mindex]
        s45 = indexMerge['SpotIndex_45'].iloc[mindex]
        s60 = indexMerge['SpotIndex_60'].iloc[mindex]

        i0  = intensity0.loc[s0,  Icol]
        i15 = intensity15.loc[s15, Icol]
        i30 = intensity30.loc[s30, Icol]
        i45 = intensity45.loc[s45, Icol]
        i60 = intensity60.loc[s60, Icol]

        x = np.array([i0, i15, i30, i45, i60], dtype=float)
        y = samples

        if (not np.all(np.isfinite(x))) or (len(np.unique(x)) < 2):
            a, b, r2 = np.nan, np.nan, np.nan
        else:
            try:
                a, b = np.polyfit(x, y, 1)
                y_hat = a * x + b
                ss_res = np.sum((y - y_hat)**2)
                ss_tot = np.sum((y - np.mean(y))**2)
                r2 = 1 - ss_res / ss_tot
            except Exception:
                a, b, r2 = np.nan, np.nan, np.nan

        result_df.loc[mindex, f'{Icol}-RE'] = a
        result_df.loc[mindex, f'{Icol}-r2'] = r2
        result_df.loc[mindex, f'{Icol}-yValue'] = b




KeyboardInterrupt: 

In [50]:

cols = [f'{c}-RE' for c in IntenCols] + [f'{c}-r2' for c in IntenCols] + [f'{c}-yValue' for c in IntenCols]
n = len(indexMerge)
result_df = pd.DataFrame(index=range(n), columns=cols, dtype=float)

# 1) 一次性取 SpotIndex（避免循环里 iloc）
s0  = indexMerge['SpotIndex_0' ].to_numpy()
s15 = indexMerge['SpotIndex_15'].to_numpy()
s30 = indexMerge['SpotIndex_30'].to_numpy()
s45 = indexMerge['SpotIndex_45'].to_numpy()
s60 = indexMerge['SpotIndex_60'].to_numpy()

# 2) 先把 5 个 intensity 表对齐到 indexMerge 的顺序（避免循环里 loc）
# 注意：这里假设 intensity0..60 的 index 就是 SpotIndex
I0  = intensity0.reindex(s0)
I15 = intensity15.reindex(s15)
I30 = intensity30.reindex(s30)
I45 = intensity45.reindex(s45)
I60 = intensity60.reindex(s60)

# y 固定
y = np.array([0, 15, 30, 45, 60], dtype=float)
y_mean = y.mean()
ss_tot = np.sum((y - y_mean) ** 2)   # 常数

for Icol in IntenCols:
    # 3) 一次性拿到所有 mindex 的 5 个 x 值：n×5
    x = np.column_stack([
        I0[Icol].to_numpy(),
        I15[Icol].to_numpy(),
        I30[Icol].to_numpy(),
        I45[Icol].to_numpy(),
        I60[Icol].to_numpy()
    ]).astype(float)

    # 有效性：每行都要 5 个都 finite，且不能全一样
    finite_mask = np.all(np.isfinite(x), axis=1)
    var_mask = (np.max(x, axis=1) - np.min(x, axis=1)) > 0
    ok = finite_mask & var_mask

    a = np.full(n, np.nan, dtype=float)
    b = np.full(n, np.nan, dtype=float)
    r2 = np.full(n, np.nan, dtype=float)

    if np.any(ok):
        x_ok = x[ok]

        # 4) 用线性回归闭式解（比 polyfit 快）：
        # slope a = cov(x,y)/var(x), intercept b = y_mean - a*x_mean
        x_mean = x_ok.mean(axis=1)
        x_centered = x_ok - x_mean[:, None]
        y_centered = y - y_mean

        cov_xy = np.sum(x_centered * y_centered[None, :], axis=1)
        var_x  = np.sum(x_centered ** 2, axis=1)

        a_ok = cov_xy / var_x
        b_ok = y_mean - a_ok * x_mean

        y_hat = a_ok[:, None] * x_ok + b_ok[:, None]
        ss_res = np.sum((y_hat - y[None, :]) ** 2, axis=1)
        r2_ok = 1 - ss_res / ss_tot

        a[ok], b[ok], r2[ok] = a_ok, b_ok, r2_ok

    # 5) 一次性写回整列（避免 result_df.loc 单元格写入）
    result_df[f'{Icol}-RE'] = a
    result_df[f'{Icol}-r2'] = r2
    result_df[f'{Icol}-yValue'] = b


In [52]:
print(result_df.iloc[:5,:9])

     57.0346-RE  58.0296-RE  59.0139-RE  67.0191-RE  71.0137-RE  71.0503-RE  \
0 -2.092564e+01   -5.470079   13.957385    8.108909    1.311504   -1.943702   
1 -2.649326e-15  -17.575835    2.776768    1.397792    0.738460   43.318733   
2 -4.213872e+00  -17.198838   -1.514525   -0.164916    3.574316   14.011314   
3  6.869407e+00  -12.792054   -1.065869    1.658418    1.494948   -8.441771   
4 -7.702156e+00   11.618542    8.754863   38.787792   -3.301581   52.292969   

   72.0091-RE  72.017-RE  73.0294-RE  
0  -15.364057 -17.388651    2.130303  
1  -17.871445  23.276861    3.292568  
2   -3.873258   3.276752   -5.289815  
3   -4.311055  -3.012225   -2.596135  
4   10.220157  15.694484    2.554996  


In [54]:
result_df.to_csv('/p2/zulab/jtian/data/SA/06_calculateConcentration/output_allCoordsClustersMerge/relativeConcentration.csv')

In [57]:
result_T = result_df.T

In [58]:
r2_rows = result_T.loc[result_T.index.str.endswith('-r2')]


In [59]:
r2_rows.index

Index(['57.0346-r2', '58.0296-r2', '59.0139-r2', '67.0191-r2', '71.0137-r2',
       '71.0503-r2', '72.0091-r2', '72.017-r2', '73.0294-r2', '74.0246-r2',
       ...
       '856.5052-r2', '859.5297-r2', '860.536-r2', '863.5618-r2',
       '864.5684-r2', '882.5853-r2', '921.5991-r2', '923.615-r2',
       '985.6046-r2', '986.6105-r2'],
      dtype='object', length=696)

In [60]:
r2_rows.shape

(696, 37936)

In [61]:
mask = r2_rows.ge(0.8).sum(axis=1) >= (r2_rows.shape[1] / 2)


In [62]:
good_r2_index = r2_rows.index[mask]

good_mz = good_r2_index.str.replace(r'-r2$', '', regex=True).tolist()


In [63]:
good_mz


['88.0401',
 '132.0297',
 '145.0614',
 '151.0607',
 '158.9249',
 '177.057',
 '231.0275',
 '272.0883',
 '277.2159',
 '304.0604',
 '306.0585',
 '306.0754',
 '421.075',
 '758.4983',
 '760.5072']

In [64]:
len(good_mz)


15